KAIA **Flight** **Data** **Cleaning &** **Feature Engineering**

This notebook prepares the King Abdulaziz International Airport flight dataset for analysis in Power BI.

Workflow
1. Load the raw CSV dataset.
2. Inspect data structure and quality.
3. Remove exact duplicate rows.
4. Flag anomalous transit durations and handle documented data-quality cases.
5. Create analysis-ready features.
6. Validate the cleaned dataset and export it to CSV.



In [19]:
import pandas as pd
import numpy as np
from google.colab import files

# Upload the raw CSV file when prompted
uploaded = files.upload()
raw_file = next(name for name in uploaded if 'dirty' in name.lower() and name.lower().endswith('.csv'))

# The source file is comma-separated
df_raw = pd.read_csv(raw_file)
df = df_raw.copy()

print(f'Raw dataset shape: {df.shape}')
display(df.head())

Saving KingAbdulazizAirport_Flight_Data_dirty.csv to KingAbdulazizAirport_Flight_Data_dirty (2).csv
Raw dataset shape: (1830, 16)


,movement,flight_id,airline,origin,transit_point,destination,take_off_time,landing_time,transit_duration_mins,total_duration_mins,aircraft_model,carbon_kg,eco_price_sar,price_status,flight_date,day
0,DEPARTURE,EY 604,Etihad,JED,Direct,AUH,05:40,09:20,0.0,160.0,Airbus A321,166000,461,typical,8/22/2026,Saturday
1,DEPARTURE,SV 570,Saudia,JED,Direct,AUH,11:40,15:25,0.0,165.0,Airbus A320,176000,461,typical,8/22/2026,Saturday
2,DEPARTURE,SV 1020,Saudia,JED,RUH,AUH,07:00,14:05,150.0,365.0,Airbus A321,223000,471,typical,8/22/2026,Saturday
3,DEPARTURE,EY 602,Etihad,JED,Direct,AUH,03:10,06:55,0.0,165.0,Airbus A321,166000,552,typical,8/22/2026,Saturday
4,DEPARTURE,QR 1189,Qatar Airways,JED,DOH,AUH,22:35,04:25,80.0,290.0,Boeing 787,152000,697,typical,8/22/2026,Saturday


In [20]:
print('=== COLUMN NAMES ===')
print(df.columns.tolist())

print('\n=== DATA TYPES ===')
display(df.dtypes.rename('data_type').to_frame())

print('\n=== MISSING VALUES ===')
display(df.isna().sum().sort_values(ascending=False).rename('missing_count').to_frame())

print('\n=== EXACT DUPLICATES ===')
print('Duplicate rows:', df.duplicated().sum())

print('\n=== NUMERICAL SUMMARY ===')
display(df.describe())

=== COLUMN NAMES ===
['movement', 'flight_id', 'airline', 'origin', 'transit_point', 'destination', 'take_off_time', 'landing_time', 'transit_duration_mins', 'total_duration_mins', 'aircraft_model', 'carbon_kg', 'eco_price_sar', 'price_status', 'flight_date', 'day']

=== DATA TYPES ===


,data_type
movement,object
flight_id,object
airline,object
origin,object
transit_point,object
destination,object
take_off_time,object
landing_time,object
transit_duration_mins,float64
total_duration_mins,float64



=== MISSING VALUES ===


,missing_count
total_duration_mins,18
transit_duration_mins,18
movement,0
flight_id,0
transit_point,0
destination,0
airline,0
origin,0
landing_time,0
take_off_time,0



=== EXACT DUPLICATES ===
Duplicate rows: 20

=== NUMERICAL SUMMARY ===


,transit_duration_mins,total_duration_mins,carbon_kg,eco_price_sar
count,1812.000000,1812.000000,1.830000e+03,1830.000000
mean,228.326711,486.568433,3.013432e+05,1468.998361
std,1402.659605,451.395006,2.438126e+05,1593.888265
min,0.000000,60.000000,5.400000e+04,249.000000
25%,0.000000,125.000000,9.800000e+04,447.000000
50%,0.000000,310.000000,2.045000e+05,985.500000
75%,155.000000,730.000000,4.687500e+05,1903.500000
max,18800.000000,2900.000000,2.333000e+06,21358.000000


In [21]:
duplicates_before = df.duplicated().sum()
df_clean = df.drop_duplicates().copy()

print('Rows before duplicate removal:', len(df))
print('Exact duplicate rows removed:', duplicates_before)
print('Rows after duplicate removal:', len(df_clean))
print('Remaining exact duplicates:', df_clean.duplicated().sum())

Rows before duplicate removal: 1830
Exact duplicate rows removed: 20
Rows after duplicate removal: 1810
Remaining exact duplicates: 0


In [22]:
# Preserve the transit-duration values before applying cleaning rules
transit_before_cleaning = df_clean['transit_duration_mins'].copy()

# Flag the documented anomalous duration before replacing it
df_clean['transit_duration_suspect'] = transit_before_cleaning.eq(18800)

# For direct flights, a missing transit duration is represented as 0 minutes
direct_missing_transit = (
    df_clean['transit_point'].eq('Direct') &
    transit_before_cleaning.isna()
)
df_clean.loc[direct_missing_transit, 'transit_duration_mins'] = 0

# Treat the documented 18,800-minute values as missing after flagging them
anomalous_transit = transit_before_cleaning.eq(18800)
df_clean.loc[anomalous_transit, 'transit_duration_mins'] = np.nan

print('Suspicious transit records flagged:', int(df_clean['transit_duration_suspect'].sum()))
print('Missing direct-flight transit durations set to 0:', int(direct_missing_transit.sum()))
print('Remaining 18,800-minute transit values:', int(df_clean['transit_duration_mins'].eq(18800).sum()))

display(df_clean.loc[df_clean['transit_duration_suspect'], [
    'flight_id', 'origin', 'transit_point', 'destination',
    'transit_duration_mins', 'transit_duration_suspect'
]].head(10))

Suspicious transit records flagged: 10
Missing direct-flight transit durations set to 0: 8
Remaining 18,800-minute transit values: 0


,flight_id,origin,transit_point,destination,transit_duration_mins,transit_duration_suspect
308,EY 612,JED,AUH,ZRH,NaN,True
562,EY 612,JED,AUH,BAH,NaN,True
739,QR 38,CDG,DOH,JED,NaN,True
930,VF 568,JED,ESB,FRA,NaN,True
1171,SV 215,JED,AMS,LHR,NaN,True
1308,F3 118,JED,Direct,RUH,NaN,True
1324,EY 401,BKK,AUH,JED,NaN,True
1442,F3 122,JED,Direct,RUH,NaN,True
1580,MS 670,JED,CAI,ORD,NaN,True
1615,TK 1958,AMS,IST,JED,NaN,True


In [23]:
# Parse the flight date consistently
df_clean['flight_date'] = pd.to_datetime(
    df_clean['flight_date'], errors='coerce'
).dt.strftime('%Y-%m-%d')

# Month field for time-based analysis
df_clean['month'] = pd.to_datetime(
    df_clean['flight_date'], errors='coerce'
).dt.strftime('%Y-%m')

# Route combines the origin and destination airport codes
df_clean['route'] = (
    df_clean['origin'].astype('string') + ' - ' +
    df_clean['destination'].astype('string')
)

# Convert total flight duration from minutes to hours
df_clean['duration_hours'] = (df_clean['total_duration_mins'] / 60).round(2)

print('Cleaned dataset shape:', df_clean.shape)
display(df_clean[[
    'flight_date', 'month', 'route', 'total_duration_mins',
    'duration_hours', 'transit_duration_suspect'
]].head())

Cleaned dataset shape: (1810, 20)


,flight_date,month,route,total_duration_mins,duration_hours,transit_duration_suspect
0,2026-08-22,2026-08,JED - AUH,160.0,2.67,False
1,2026-08-22,2026-08,JED - AUH,165.0,2.75,False
2,2026-08-22,2026-08,JED - AUH,365.0,6.08,False
3,2026-08-22,2026-08,JED - AUH,165.0,2.75,False
4,2026-08-22,2026-08,JED - AUH,290.0,4.83,False


In [24]:
print('=== VALIDATION SUMMARY ===')
print('Rows:', len(df_clean))
print('Columns:', len(df_clean.columns))
print('Remaining exact duplicates:', int(df_clean.duplicated().sum()))
print('Suspicious transit records:', int(df_clean['transit_duration_suspect'].sum()))
print('\nMissing values by column:')
display(df_clean.isna().sum().sort_values(ascending=False).rename('missing_count').to_frame())

# Expected structure for this dataset version
assert len(df_clean) == 1810, 'Unexpected row count: review the cleaning steps.'
assert len(df_clean.columns) == 20, 'Unexpected column count: review feature engineering.'
assert df_clean.duplicated().sum() == 0, 'Exact duplicates remain.'
assert int(df_clean['transit_duration_suspect'].sum()) == 10, 'Review suspicious-transit rule.'

print('\nValidation checks passed.')

=== VALIDATION SUMMARY ===
Rows: 1810
Columns: 20
Remaining exact duplicates: 0
Suspicious transit records: 10

Missing values by column:


,missing_count
transit_duration_mins,20
total_duration_mins,18
duration_hours,18
movement,0
origin,0
airline,0
flight_id,0
transit_point,0
landing_time,0
take_off_time,0



Validation checks passed.


In [25]:
output_file = 'King_Abdulaziz_Airport_Clean.csv'
df_clean.to_csv(output_file, index=False)

print('Export completed:', output_file)
print('Exported shape:', df_clean.shape)

# Download the cleaned CSV from Google Colab
files.download(output_file)

Export completed: King_Abdulaziz_Airport_Clean.csv
Exported shape: (1810, 20)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>